# 5 · Server-side gating · the `bm:servable` bitmap + Lua script

<img src="https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120" alt="Redis"/>

<a href="https://colab.research.google.com/github/redis-field-engineering/redis-dsp-demo/blob/main/notebooks/05_bitmap_gate_lua.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part of the Redis DSP candidate-generation demo. The full sequence walks the bid path from naive to fast across six notebooks; this is `05_bitmap_gate_lua.ipynb`.

**To run in Colab:** click the badge above, then *Runtime → Run all*. The setup cells below clone the repo, install dependencies, start a Redis Stack server, and load the synthetic dataset.

**To run locally:** make sure the docker-compose stack is up (`make up` from the repo root). The setup cells detect a local environment and skip the Colab-specific steps.

## Setup

These five cells prepare the environment. They are idempotent — safe to re-run, safe in either Colab or local. On Colab the first run takes about 60–90 seconds (pip install + apt install + dataset generation). Subsequent runs are near-instant because everything is cached.

In [1]:
# Setup 1/5 · clone the repo (Colab only).
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("pyproject.toml"):
    print("Cloning https://github.com/redis-field-engineering/redis-dsp-demo ...")
    os.system("git clone -q https://github.com/redis-field-engineering/redis-dsp-demo.git _repo")
    os.system("cp -R _repo/. ./")
    os.system("rm -rf _repo")
    print("Repo cloned.")
elif not IN_COLAB:
    print("Local environment detected — skipping clone.")
else:
    print("Repo already present.")

Local environment detected — skipping clone.


In [2]:
# Setup 2/5 · install Python dependencies (Colab only).
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.system(
        f'{sys.executable} -m pip install -q '
        '"redis[hiredis]>=5.2.0" "pydantic>=2.9.0" "pandas>=2.2.0" "pyarrow>=18.0.0"'
    )
    print("Dependencies installed.")
else:
    print("Local environment detected — skipping pip install (assumes deps are already installed).")

Local environment detected — skipping pip install (assumes deps are already installed).


In [3]:
# Setup 3/5 · install and start Redis Stack (Colab only).
import os, sys, shutil
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if shutil.which("redis-stack-server") is None:
        print("Installing redis-stack-server ...")
        os.system(
            'curl -fsSL https://packages.redis.io/gpg | '
            'sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg'
        )
        os.system(
            'echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] '
            'https://packages.redis.io/deb $(lsb_release -cs) main" '
            '| sudo tee /etc/apt/sources.list.d/redis.list > /dev/null'
        )
        os.system("sudo apt-get update -qq > /dev/null 2>&1")
        os.system("sudo apt-get install -qq -y redis-stack-server > /dev/null 2>&1")
    os.system("redis-stack-server --daemonize yes > /dev/null 2>&1")
    print("redis-stack-server started on :6379")
else:
    print("Local environment detected — skipping Redis install (expecting docker-compose Redis at localhost:6381).")

Local environment detected — skipping Redis install (expecting docker-compose Redis at localhost:6381).


In [4]:
# Setup 4/5 · choose the Redis URL.
import os, sys
IN_COLAB = "google.colab" in sys.modules
default_port = "6379" if IN_COLAB else "6381"
REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = os.getenv("REDIS_PORT", default_port)
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")
auth = f":{REDIS_PASSWORD}@" if REDIS_PASSWORD else ""
REDIS_URL = f"redis://{auth}{REDIS_HOST}:{REDIS_PORT}/0"
os.environ["DEMO_REDIS_URL"] = REDIS_URL
print(f"Redis URL: {REDIS_URL}")

Redis URL: redis://localhost:6381/0


In [5]:
# Setup 5/5 · generate and load the synthetic dataset (only if Redis is empty).
import sys, subprocess
from pathlib import Path

# Find the repo root so we can run `python -m data.synthetic` reliably.
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from redis import Redis
client = Redis.from_url(REDIS_URL, decode_responses=True)
if not client.ping():
    raise RuntimeError(f"Redis at {REDIS_URL} did not answer PING")

if client.exists("meta:dataset_loaded"):
    print(
        f"Dataset already loaded: "
        f"{client.get('meta:user_count')} users, "
        f"{client.get('meta:campaign_count')} campaigns."
    )
else:
    print("Generating synthetic dataset (~30 seconds) ...")
    subprocess.run(
        [sys.executable, "-m", "data.synthetic",
         "--output", "data/generated/synthetic",
         "--num-users", "4000",
         "--num-campaigns", "2500",
         "--num-interactions", "120000",
         "--feature-count", "12"],
        cwd=_repo_root, check=True,
    )
    print("Loading dataset into Redis ...")
    subprocess.run(
        [sys.executable, "-m", "data.load_redis",
         "--redis-url", REDIS_URL,
         "--dataset-dir", "data/generated/synthetic"],
        cwd=_repo_root, check=True,
    )
    print(
        f"Done. {client.get('meta:user_count')} users, "
        f"{client.get('meta:campaign_count')} campaigns."
    )

Dataset already loaded: 4000 users, 2500 campaigns.


## Walkthrough

From here on the notebook is the demonstration.

In [6]:
# Locate the repo root so `notebooks._demo_setup` is importable regardless
# of where the kernel was launched (the package layout requires the repo
# root on sys.path).
import sys
from pathlib import Path
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from notebooks._demo_setup import connect_redis, StepTimer
client = connect_redis()

connected to redis://localhost:6381/0
  users=4000  campaigns=2500  precompute_version=v17_2500_12


## What's in `bm:servable`

A single global Redis BITMAP. Each campaign has a bit position derived
from its ID (`c00042` → bit 42). The bit is set iff the campaign is
both `pacing_status == 'active'` **and** `spent_today_usd <
daily_budget_usd`. Updated alongside any pacing-state write.


In [7]:
bitcount = client.bitcount('bm:servable')
total_campaigns = int(client.get('meta:campaign_count'))
print(f'bm:servable bitcount = {bitcount}')
print(f'total campaigns      = {total_campaigns}')
print(f'fraction servable    = {bitcount/total_campaigns:.1%}')

# Sample: is c00042 servable right now?
bit_index = 42
print(f'\nGETBIT bm:servable {bit_index} = {client.getbit("bm:servable", bit_index)}')

bm:servable bitcount = 2158
total campaigns      = 2500
fraction servable    = 86.3%

GETBIT bm:servable 42 = 1


## The Lua script

This is the actual script registered by `app/repository.py`. It's small
on purpose — Redis Lua is single-threaded against the keyspace, so the
script must be O(N) in the candidate list and nothing fancier.


In [8]:
import inspect
from app.repository import RedisRepository

# Re-extract the script source from the class so the notebook stays in sync
# with whatever ships in the repo.
source = inspect.getsource(RedisRepository.__init__)
script_start = source.index('register_script(')
script_end = source.index('"""', source.index('"""', script_start) + 3)
print(source[script_start:script_end + 3])

register_script(
            """
            local payload = redis.call('GET', KEYS[1])
            if not payload then
                return {}
            end
            local candidate_ids = cjson.decode(payload)
            local max_results = tonumber(ARGV[1])
            local gated = {}
            for _, campaign_id in ipairs(candidate_ids) do
                local bit_index = tonumber(string.sub(campaign_id, 2))
                if redis.call('GETBIT', KEYS[2], bit_index) == 1 then
                    table.insert(gated, campaign_id)
                    if #gated >= max_results then
                        break
                    end
                end
            end
            return gated
            """


Walking through the script:

1. `GET KEYS[1]` — load the precomputed `aud:{maid_id}` JSON list.
2. Parse it.
3. For each candidate ID, `GETBIT KEYS[2] bit_index` against `bm:servable`.
4. Keep only the ones where the bit is set, capped at `ARGV[1]` results.

Everything happens inside Redis on a single shard with the candidate-list
key. From the bid engine's perspective: one round trip, returns the gated
list.


## Running the gate end-to-end

This is the `hybrid_bitmap_gating` mode.


In [9]:
from app.candidate import filter_campaigns_for_user
from app.models import ScoringProfile, Campaign
from app.ranking import rerank_campaigns

# Use the prototype's repository class — it already has the script registered.
from app.repository import RedisRepository
repo = RedisRepository('redis://localhost:6381/0')

IDENTITY_TOKEN = 'id_00042_01'
timer = StepTimer()

with timer.step('identity_resolution'):
    maid_id, _ = repo.resolve_identity(IDENTITY_TOKEN)

with timer.step('hot_profile_fetch'):
    scoring, _ = repo.fetch_scoring_profile(maid_id)

with timer.step('bitmap_gated_candidate_fetch'):
    # One round trip — Lua does the AUD-load + per-candidate GETBIT inline.
    gated_ids, _ = repo.fetch_bitmap_gated_user_candidates(maid_id, limit=200)

with timer.step('campaign_fetch_pipelined'):
    campaigns, _ = repo.fetch_campaigns(gated_ids)

with timer.step('fcap_fetch'):
    fcap_counts, _ = repo.fetch_frequency_caps(maid_id, gated_ids)

with timer.step('frequency_only_filter'):
    eligible = [
        c for c in campaigns
        if fcap_counts.get(c.campaign_id, 0) < c.frequency_cap
    ]

with timer.step('rerank'):
    top_5 = rerank_campaigns(scoring, eligible, top_k=5)

print(f'maid_id            = {maid_id}')
print(f'aud candidates     = (input to bitmap gate)')
print(f'bitmap-gated       = {len(gated_ids)}  (returned in 1 round trip)')
print(f'eligible after fcap = {len(eligible)}')
print(f'top 5: {[(r.campaign_id, round(r.score, 4)) for r in top_5]}')
print()
print(timer.summary())

maid_id            = maid_00042
aud candidates     = (input to bitmap gate)
bitmap-gated       = 21  (returned in 1 round trip)
eligible after fcap = 17
top 5: [('c01011', 5.8237), ('c00848', 4.8107), ('c01551', 4.4065), ('c01222', 4.2812), ('c02229', 4.1223)]

             identity_resolution    2.078 ms
               hot_profile_fetch    0.228 ms
    bitmap_gated_candidate_fetch    1.326 ms
        campaign_fetch_pipelined    0.942 ms
                      fcap_fetch    0.212 ms
           frequency_only_filter    0.004 ms
                          rerank    0.069 ms
--------------------------------------------
                           TOTAL    4.859 ms


## What the bitmap saves us

Compared to `precomputed_segment`:

- the per-candidate `campaign_state:` fanout disappears (fewer round trips
  *and* less app-side state-merge logic),
- the candidate count drops further because the bitmap pre-filters out
  campaigns that have gone unservable since the precompute was built.

On a tuned VM, `hybrid_bitmap_gating` lands at `~1.9 ms` p50 — the fastest
mode that doesn't evaluate the per-campaign taxonomy filter.

But it doesn't evaluate the per-campaign taxonomy filter, which is the
question the customer's PDF actually asks about. That's notebook 6.
